# Comparative Study on Real Sentinel-2 Multispectral Imagery & RGI 14 Glacier Outlines

This notebook trains and evaluates **U-Net**, **Attention U-Net**, and **SUNet (Swin-UNet)** on real 8-channel Sentinel-2 satellite data and Randolph Glacier Inventory (RGI 14: South Asia West):
1. **Mounts Google Drive**: Reads your 12 Sentinel-2 GeoTIFF tiles and the unzipped RGI 14 folder directly from `MyDrive/Minor_Project`.
2. **Geospatial Rasterization**: Automatically matches the RGI glacier shapefile with the 10m Sentinel-2 grid to generate ground-truth binary masks.
3. **8-Channel Patch Extraction**: Extracts $256 \times 256$ patches normalized across all 8 bands (`B2, B3, B4, B8, B11, B12, NDSI, NDVI`).
4. **Comparative GPU Training**: Trains all 3 models for 25 epochs with Hybrid BCE + Dice Loss.
5. **Results & Visualizations**: Automatically exports comparative metrics (`.csv`), bar charts, side-by-side prediction maps, and model checkpoints back into your `Minor_Project` Drive folder.

## 1. GPU Check & Geospatial Libraries Installation

In [ ]:
# Install geospatial processing packages
!pip install -q rasterio geopandas shapely

import os
import glob
import zipfile
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Switch to GPU under Runtime -> Change runtime type.")

## 2. Mount Google Drive & Locate `Minor_Project` Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Search for Minor_Project folder in Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Minor_Project"
if not os.path.exists(DRIVE_DIR):
    matches = [p for p in glob.glob("/content/drive/MyDrive/*Minor*") if os.path.isdir(p)]
    if matches:
        DRIVE_DIR = matches[0]
    else:
        raise FileNotFoundError(f"Could not find '{DRIVE_DIR}'. Please check your Drive folder name.")

print(f"Working Directory: {DRIVE_DIR}")

# 1. Locate 12 GeoTIFF tiles
tif_files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.tif")))
print(f"Found {len(tif_files)} Sentinel-2 GeoTIFF tiles:")
for f in tif_files[:3]:
    print("  -", os.path.basename(f))
if len(tif_files) > 3:
    print(f"  ... and {len(tif_files)-3} more.")

# 2. Locate RGI 14 Shapefile (folder or zip)
shp_files = glob.glob(os.path.join(DRIVE_DIR, "**/*.shp"), recursive=True)
if not shp_files:
    # Check if there is a zip file to extract
    zip_candidates = glob.glob(os.path.join(DRIVE_DIR, "*14*.zip"))
    if zip_candidates:
        print("Found zip archive, extracting...")
        with zipfile.ZipFile(zip_candidates[0], 'r') as z:
            z.extractall("/content/rgi14")
        shp_files = glob.glob("/content/rgi14/**/*.shp", recursive=True)

assert len(shp_files) > 0, "Could not find .shp file in Minor_Project!"
RGI_SHP = shp_files[0]
print(f"Found RGI 14 Shapefile: {RGI_SHP}")

## 3. Load RGI 14 Glacier Shapefile

In [ ]:
import geopandas as gpd

print("Loading RGI glacier polygons...")
gdf = gpd.read_file(RGI_SHP)
print(f"Loaded {len(gdf):,} glacier outlines.")
print(f"Coordinate Reference System (CRS): {gdf.crs}")

## 4. Extract Real 8-Channel 256x256 Patches & Rasterize Ground Truth Masks
Loops through the 12 Sentinel-2 GeoTIFF tiles, slices them into $256 \times 256$ arrays, and rasterizes matching RGI glacier polygons to binary masks ($1 =$ Glacier, $0 =$ Non-glacier).

In [ ]:
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window

# Local fast scratch space on Colab SSD
PATCH_DIR = "/content/real_dataset"
TRAIN_DIR = os.path.join(PATCH_DIR, "train")
VAL_DIR = os.path.join(PATCH_DIR, "val")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

PATCH_SIZE = 256
STRIDE = 256
MAX_PATCHES_TOTAL = 1200  # Total patches across all 12 tiles (sufficient for fast and accurate training)

patch_count = 0
print("Extracting 8-channel Sentinel-2 patches & rasterizing RGI masks...")

for tif_path in tif_files:
    if patch_count >= MAX_PATCHES_TOTAL:
        break
        
    tile_name = os.path.basename(tif_path)
    print(f"\nProcessing tile: {tile_name}")
    
    with rasterio.open(tif_path) as src:
        H, W = src.height, src.width
        crs = src.crs
        bounds = src.bounds
        
        # Reproject polygons to match raster CRS if needed
        if gdf.crs != crs:
            gdf_reproj = gdf.to_crs(crs)
        else:
            gdf_reproj = gdf
            
        intersecting_glaciers = gdf_reproj.cx[bounds.left:bounds.right, bounds.bottom:bounds.top]
        if len(intersecting_glaciers) == 0:
            print(f"  Tile has no intersecting glaciers. Skipping.")
            continue
            
        print(f"  Intersecting glaciers in this tile: {len(intersecting_glaciers)}")
        
        y_coords = list(range(0, H - PATCH_SIZE, STRIDE))
        x_coords = list(range(0, W - PATCH_SIZE, STRIDE))
        np.random.seed(42)
        np.random.shuffle(y_coords)
        np.random.shuffle(x_coords)
        
        tile_patches = 0
        for y in y_coords[:35]:
            if patch_count >= MAX_PATCHES_TOTAL:
                break
            for x in x_coords[:35]:
                if patch_count >= MAX_PATCHES_TOTAL:
                    break
                    
                window = Window(x, y, PATCH_SIZE, PATCH_SIZE)
                patch_data = src.read(window=window).astype(np.float32)  # shape: (8, 256, 256)
                
                # Discard empty/nodata windows
                if np.isnan(patch_data).all() or (patch_data == 0).all():
                    continue
                patch_data = np.nan_to_num(patch_data, nan=0.0)
                
                win_transform = rasterio.windows.transform(window, src.transform)
                win_bounds = rasterio.windows.bounds(window, src.transform)
                win_glaciers = intersecting_glaciers.cx[win_bounds[0]:win_bounds[2], win_bounds[1]:win_bounds[3]]
                
                if len(win_glaciers) > 0:
                    shapes = [(geom, 1) for geom in win_glaciers.geometry if geom.is_valid]
                    mask = rasterize(shapes, out_shape=(PATCH_SIZE, PATCH_SIZE), transform=win_transform, fill=0, dtype=np.uint8)
                else:
                    mask = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)
                
                glacier_ratio = np.mean(mask)
                # Include patches containing glacier or nearby surrounding mountainous terrain
                if glacier_ratio > 0.02 or (tile_patches % 3 == 0):
                    # Min-Max normalize each channel
                    for c in range(patch_data.shape[0]):
                        c_min = patch_data[c].min()
                        c_max = patch_data[c].max()
                        if c_max - c_min > 1e-6:
                            patch_data[c] = (patch_data[c] - c_min) / (c_max - c_min)
                    
                    # 80% Train, 20% Validation split
                    split = TRAIN_DIR if (patch_count % 5 != 0) else VAL_DIR
                    np.save(os.path.join(split, f"patch_{patch_count:05d}_patch.npy"), patch_data)
                    np.save(os.path.join(split, f"patch_{patch_count:05d}_mask.npy"), mask.astype(np.float32))
                    patch_count += 1
                    tile_patches += 1

train_patches = len(glob.glob(os.path.join(TRAIN_DIR, "*_patch.npy")))
val_patches = len(glob.glob(os.path.join(VAL_DIR, "*_patch.npy")))
print(f"\nDataset Generation Complete!")
print(f"  Training Patches:   {train_patches}")
print(f"  Validation Patches: {val_patches}")

## 5. PyTorch Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class RealGlacierDataset(Dataset):
    def __init__(self, folder):
        self.patch_files = sorted(glob.glob(os.path.join(folder, "*_patch.npy")))
        self.mask_files = [f.replace("_patch.npy", "_mask.npy") for f in self.patch_files]
        
    def __len__(self):
        return len(self.patch_files)
        
    def __getitem__(self, idx):
        patch = np.load(self.patch_files[idx])
        mask = np.load(self.mask_files[idx])
        return torch.from_numpy(patch).float(), torch.from_numpy(mask).float().unsqueeze(0)

train_ds = RealGlacierDataset(TRAIN_DIR)
val_ds = RealGlacierDataset(VAL_DIR)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

print(f"DataLoaders ready: {len(train_loader)} training batches, {len(val_loader)} validation batches.")

## 6. Model Architectures (U-Net, Attention U-Net, SUNet)
Configured for 8 multispectral input channels (`B2, B3, B4, B8, B11, B12, NDSI, NDVI`).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True)
        )
    def forward(self, x): return self.conv(x)

# 1. Standard 8-Channel U-Net
class UNet8Channel(nn.Module):
    def __init__(self, in_c=8, out_c=1):
        super().__init__()
        self.inc = DoubleConv(in_c, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.cup1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.cup2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.cup3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.cup4 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_c, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        d1 = self.cup1(torch.cat([self.up1(x5), x4], 1))
        d2 = self.cup2(torch.cat([self.up2(d1), x3], 1))
        d3 = self.cup3(torch.cat([self.up3(d2), x2], 1))
        d4 = self.cup4(torch.cat([self.up4(d3), x1], 1))
        return self.outc(d4)

# 2. Attention 8-Channel U-Net
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.Wx = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        psi_conv = nn.Conv2d(F_int, 1, 1)
        nn.init.constant_(psi_conv.bias, 1.0)
        self.psi = nn.Sequential(psi_conv, nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(True)
    def forward(self, g, x):
        g1 = self.Wg(g)
        x1 = self.Wx(x)
        if g1.shape[2:] != x1.shape[2:]: g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=True)
        return x * self.psi(self.relu(g1 + x1))

class AttentionUNet8Channel(nn.Module):
    def __init__(self, in_c=8, out_c=1):
        super().__init__()
        self.inc = DoubleConv(in_c, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        self.ag1 = AttentionGate(512, 512, 256)
        self.ag2 = AttentionGate(256, 256, 128)
        self.ag3 = AttentionGate(128, 128, 64)
        self.ag4 = AttentionGate(64, 64, 32)
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.cup1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.cup2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.cup3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.cup4 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_c, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        u1 = self.up1(x5)
        d1 = self.cup1(torch.cat([u1, self.ag1(u1, x4)], 1))
        u2 = self.up2(d1)
        d2 = self.cup2(torch.cat([u2, self.ag2(u2, x3)], 1))
        u3 = self.up3(d2)
        d3 = self.cup3(torch.cat([u3, self.ag3(u3, x2)], 1))
        u4 = self.up4(d3)
        d4 = self.cup4(torch.cat([u4, self.ag4(u4, x1)], 1))
        return self.outc(d4)

# 3. SUNet 8-Channel (Swin-UNet)
class SwinBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, x):
        B, C, H, W = x.shape
        xf = x.flatten(2).transpose(1, 2)
        att, _ = self.attn(self.norm1(xf), self.norm1(xf), self.norm1(xf))
        xf = xf + att
        xf = xf + self.mlp(self.norm2(xf))
        return xf.transpose(1, 2).view(B, C, H, W)

class SUNet8Channel(nn.Module):
    def __init__(self, in_c=8, out_c=1):
        super().__init__()
        self.inc = DoubleConv(in_c, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024), SwinBlock(1024, 8))
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.cup1 = nn.Sequential(DoubleConv(1024, 512), SwinBlock(512, 4))
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.cup2 = nn.Sequential(DoubleConv(512, 256), SwinBlock(256, 4))
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.cup3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.cup4 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_c, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        d1 = self.cup1(torch.cat([self.up1(x5), x4], 1))
        d2 = self.cup2(torch.cat([self.up2(d1), x3], 1))
        d3 = self.cup3(torch.cat([self.up3(d2), x2], 1))
        d4 = self.cup4(torch.cat([self.up4(d3), x1], 1))
        return self.outc(d4)

print("8-Channel Model architectures (UNet, AttentionUNet, SUNet) defined successfully.")

## 7. Comparative Benchmark Training Loop (25 Epochs)
Trains each model on GPU using Hybrid BCE + Dice Loss (alpha = 0.5) and evaluates at the optimal threshold (0.70).

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1)
        t = targets.view(-1)
        return 1.0 - (2.0 * (p * t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)

class HybridLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return self.alpha * self.bce(logits, targets) + (1 - self.alpha) * self.dice(logits, targets)

def evaluate_model(model, loader, device, th=0.70):
    model.eval()
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = (torch.sigmoid(model(x)) >= th).float()
            tp += (preds * y).sum().item()
            fp += (preds * (1 - y)).sum().item()
            fn += ((1 - preds) * y).sum().item()
            tn += ((1 - preds) * (1 - y)).sum().item()
    eps = 1e-8
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    iou = tp / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    return dice, iou, prec, rec

# Create Output Directory directly inside your Drive Minor_Project folder
OUTPUT_RESULTS_DIR = os.path.join(DRIVE_DIR, "results_real_sentinel2")
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

models_dict = {
    "U-Net (Baseline)": UNet8Channel(in_c=8, out_c=1),
    "Attention U-Net": AttentionUNet8Channel(in_c=8, out_c=1),
    "SUNet (Swin-UNet)": SUNet8Channel(in_c=8, out_c=1)
}

benchmark_results = []
trained_models = {}
epochs = 25
lr = 2e-4

for name, m in models_dict.items():
    print(f"\n===================================================")
    print(f"   Training: {name} on Real Sentinel-2 ({epochs} Epochs)")
    print(f"===================================================")
    m = m.to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    crit = HybridLoss(alpha=0.5)
    best_dice = 0.0
    best_metrics = (0, 0, 0, 0)
    
    for epoch in range(1, epochs + 1):
        m.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            loss = crit(m(bx), by)
            loss.backward()
            opt.step()
            train_loss += loss.item()
            
        d, i, p, r = evaluate_model(m, val_loader, device, th=0.70)
        if d > best_dice:
            best_dice = d
            best_metrics = (d, i, p, r)
            torch.save(m.state_dict(), os.path.join(OUTPUT_RESULTS_DIR, f"{name.lower().replace(' ', '_')}_best.pth"))
            
        if epoch % 5 == 0 or epoch == epochs:
            print(f"   Epoch {epoch:02d}/{epochs} -> Dice: {d:.4f} | IoU: {i:.4f} | Precision: {p:.4f} | Recall: {r:.4f}")
    
    trained_models[name] = m
    params_m = sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
    benchmark_results.append({
        "Model Architecture": name,
        "Parameters (M)": round(params_m, 2),
        "Dice Score": round(best_metrics[0], 4),
        "IoU Score": round(best_metrics[1], 4),
        "Precision": round(best_metrics[2], 4),
        "Recall": round(best_metrics[3], 4)
    })

df_results = pd.DataFrame(benchmark_results)
print("\n===================================================")
print("       REAL SATELLITE COMPARATIVE BENCHMARK        ")
print("===================================================")
display(df_results)

# Save Results CSV to Google Drive
csv_out = os.path.join(OUTPUT_RESULTS_DIR, "comparative_metrics_real_sentinel2.csv")
df_results.to_csv(csv_out, index=False)
print(f"Saved results table directly to: {csv_out}")

## 8. Generate & Save Performance Visualizations Directly to Drive

In [ ]:
# 1. Plot Metrics Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_results["Model Architecture"]))
width = 0.2

ax.bar(x - 1.5*width, df_results["Dice Score"], width, label="Dice Score", color='#1f77b4')
ax.bar(x - 0.5*width, df_results["IoU Score"], width, label="IoU Score", color='#2ca02c')
ax.bar(x + 0.5*width, df_results["Precision"], width, label="Precision", color='#ff7f0e')
ax.bar(x + 1.5*width, df_results["Recall"], width, label="Recall", color='#d62728')

ax.set_ylabel("Score")
ax.set_title("Comparative Model Performance on Real Sentinel-2 (Ladakh Himalaya)")
ax.set_xticks(x)
ax.set_xticklabels(df_results["Model Architecture"], fontweight="bold")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

chart_path = os.path.join(OUTPUT_RESULTS_DIR, "comparative_metrics_chart.png")
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"Saved metrics chart to: {chart_path}")

# 2. Side-by-Side Visual Segmentation Overlays
sample_patch, sample_mask = val_ds[0]
bx = sample_patch.unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
patch_np = sample_patch.numpy()
# False color composite (B8 NIR, B4 Red, B3 Green)
rgb = np.stack([patch_np[3], patch_np[2], patch_np[1]], axis=-1)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

axes[0].imshow(rgb)
axes[0].set_title("Sentinel-2 False Color (NIR/R/G)")
axes[0].axis('off')

axes[1].imshow(sample_mask.squeeze().numpy(), cmap='gray')
axes[1].set_title("RGI Ground Truth")
axes[1].axis('off')

for idx, (name, m) in enumerate(trained_models.items()):
    m.eval()
    with torch.no_grad():
        prob = torch.sigmoid(m(bx)).squeeze().cpu().numpy()
        pred = (prob >= 0.70).astype(np.uint8)
    axes[2 + idx].imshow(pred, cmap='Blues')
    d_val = df_results.loc[df_results['Model Architecture'] == name, 'Dice Score'].values[0]
    axes[2 + idx].set_title(f"{name}\n(Dice: {d_val:.4f})")
    axes[2 + idx].axis('off')

plt.tight_layout()
vis_path = os.path.join(OUTPUT_RESULTS_DIR, "comparative_predictions_overlay.png")
plt.savefig(vis_path, dpi=300)
plt.show()
print(f"Saved prediction overlays to: {vis_path}")
print(f"\nAll outputs safely saved to your Google Drive folder: '{OUTPUT_RESULTS_DIR}'!")